# Test des modules serial et pyserial

Elaboration d'un algorythme de lecture série des données de la carte

In [1]:
import serial
import serial.tools.list_ports
import matplotlib.pyplot as plt

In [2]:
ports = serial.tools.list_ports.comports()

open_ports = ""
descr = ""

for port, desc, hwid in sorted(ports):
        ##print("{}: {} [{}]".format(port, desc, hwid))
        ##print("{}".format(port))
        descr = "{}".format(desc)
        if (descr == "STM32 STLink - ST-Link VCP Ctrl"):
            open_ports = "{}".format(port)

print("Open ports : ", open_ports)

data_serial = serial.Serial(
    port=open_ports,
    baudrate=115200,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
    bytesize=serial.EIGHTBITS

)
data_serial.writeTimeout = 0 
data_serial.isOpen()

Open ports :  /dev/ttyACM0


True

In [3]:
##data_serial.close()
##data_serial.isOpen()

In [4]:
print(data_serial)

Serial<id=0x7f94e78566b0, open=True>(port='/dev/ttyACM0', baudrate=115200, bytesize=8, parity='N', stopbits=1, timeout=None, xonxoff=False, rtscts=False, dsrdtr=False)


### RX

In [5]:
ID_RX_START = b'0x01'
ID_RX_STOP = b'0x02'
ID_RX_PWM_START = b'0x03'
ID_RX_PWM_STOP = b'0x04'
ID_RX_DUTY_SET = b'0x05'
ID_RX_DIR = b'0x06'
ID_RX_VOLTAGE_SET = b'0x07'

ID_RX_SEND_KP = b'0x08'
ID_RX_SET_KP = b'0x09'

ID_RX_SEND_KP = b'0x0A'
ID_RX_SET_KP = b'0x0B'
ID_RX_SEND_KI = b'0x0C'
ID_RX_SET_KI = b'0x0D'
ID_RX_SEND_KD = b'0x0E'
ID_RX_SET_KD = b'0x0F'

In [6]:
#data_serial.write(b'hello')

def define_trame(msg_id, value, max_size):
    trame = b''
    byte_value = b''
    #byte_filler = b'0000000000000000' #filler of 16 bytes 
    #print(type(msg_id), type(value), max_size)
    byte_value = bytes(str(value),'utf-8')
    float_byte_size = len(byte_value)
    index_max_byte = max_size-len(msg_id)-2
    trame = msg_id+byte_value[0:index_max_byte]+b';'+bytes('\n', 'utf-8')
    return trame

In [7]:
trame_test = define_trame(ID_RX_START, 0.10101, 18)
print(trame_test, "len :",len(trame_test))

b'0x010.10101;\n' len : 13


In [8]:
### FONCTION ENVOI

MSG_MAX_SIZE = 16

def usart_send_start():
    data_serial.write(define_trame(ID_RX_START, 1, MSG_MAX_SIZE))
    
def usart_send_stop():
    data_serial.write(define_trame(ID_RX_STOP, 1, MSG_MAX_SIZE))
    
def usart_send_pwmStart():
    data_serial.write(define_trame(ID_RX_PWM_START, 1, MSG_MAX_SIZE))
    
def usart_send_pwmStop():
    data_serial.write(define_trame(ID_RX_PWM_STOP, 1, MSG_MAX_SIZE))
    
def usart_send_dutyValue(duty_value):
    data_serial.write(define_trame(ID_RX_DUTY_SET, duty_value, MSG_MAX_SIZE))
    
def usart_send_direction(dir_value):
    data_serial.write(define_trame(ID_RX_DIR, dir_value, MSG_MAX_SIZE))
    
def usart_send_voltageValue(voltage_value):
    data_serial.write(define_trame(ID_RX_VOLTAGE_SET, voltage_value, MSG_MAX_SIZE))
    
def usart_send_kpAsk(kp_value):
    data_serial.write(define_trame(ID_RX_SET_KP, kp_value, MSG_MAX_SIZE))
    
def usart_send_kpValue():
    data_serial.write(define_trame(ID_RX_SEND_KP, 1, MSG_MAX_SIZE))
    
def usart_send_kiAsk(ki_value):
    data_serial.write(define_trame(ID_RX_SET_KI, ki_value, MSG_MAX_SIZE))
    
def usart_send_kiValue():
    data_serial.write(define_trame(ID_RX_SEND_KI, 1, MSG_MAX_SIZE))
    
def usart_send_kdAsk(kd_value):
    data_serial.write(define_trame(ID_RX_SET_KD, kd_value, MSG_MAX_SIZE))
    
def usart_send_kdValue():
    data_serial.write(define_trame(ID_RX_SEND_KD, 1, MSG_MAX_SIZE))

In [11]:
usart_send_voltageValue(23.0)

In [39]:
data_table = []

def read_data(nline):
    it = 0
    data_table = []
    while (it != nline):
        # Read data out of the buffer until a carraige return / new line is found
        data = (data_serial.readline()).decode()
        data = data.split(";")
        print(data)
        
        data_table.append(data)
        it += 1;
    print("out")
    return data_table

In [ ]:
data_test = read_data(10)

In [ ]:
print(data_test)

In [ ]:
def compute_data_table(dt):
    ndt = []
    tables = []
    
    for data_line in dt:
        if (((len(data_line)-1) % 2 == 0) and (len(data_line) > 1)):
            data_line[0] = data_line[0].lstrip('\r') #supprime le \r caractere
            
            for i in range(0, len(data_line)):
                if ((i%2) == 1):
                    data_line[i] = float(data_line[i])
        ndt.append(data_line)
    return ndt


In [ ]:
computed_data = (compute_data_table(data_test))
print(computed_data)

In [ ]:
def time_getTable(dt):
    ndt = []
    for data_line in dt:
        if (((len(data_line)-1) % 2 == 0) and (len(data_line) > 1)):
            for k in range(len(data_line)):
                if (data_line[k] == "0x1"):
                    ndt.append(data_line[k+1]/1000)
                    
    return ndt

def pwm_getTable(dt):
    ndt = []
    for data_line in dt:
        if (((len(data_line)-1) % 2 == 0) and (len(data_line) > 1)):
            ##print(data_line)
            for k in range(len(data_line)):
                if (data_line[k] == "0x11"):
                    ndt.append(data_line[k+1])
                    
    return ndt

def adc_getTable(dt):
    ndt = []
    for data_line in dt:
        if (((len(data_line)-1) % 2 == 0) and (len(data_line) > 1)):
            for k in range(len(data_line)):
                if (data_line[k] == "0x10"):
                    ndt.append(data_line[k+1])
                    
    return ndt

def speed_getTable(dt):
    ndt = []
    for data_line in dt:
        #print("DATA LINE :", data_line, len(data_line))
        if (((len(data_line)-1) % 2 == 0) and (len(data_line) > 1)):
            for k in range(len(data_line)):
                if (data_line[k] == "0x12"):
                    ndt.append(data_line[k+1])
                    
    return ndt

In [ ]:
time_table = time_getTable(computed_data)
speed_table = speed_getTable(computed_data)
adc_table = adc_getTable(computed_data)
pwm_table = pwm_getTable(computed_data)

In [ ]:
print(adc_table)

In [ ]:
print(time_table)

In [ ]:
time_error_table = []
command_error= 0
for i in range(len(time_table)):
    if (i == 0):
        time_error_table.append(time_table[i]-time_table[i])
        command_error += time_table[i]-time_table[i]
    else:
        time_error_table.append(time_table[i]-time_table[i-1]-1.0)
        command_error += time_table[i]-time_table[i-1]-1.0

command_error /= len(time_error_table)
print(100%(1+command_error)/1)
##plt.step(x= time_table[1:len(time_table)], y= time_error_table[1:len(time_table)])


In [ ]:
plt.plot(time_table,pwm_table, label = "Commande")
plt.plot(time_table, speed_table, label = "Vitesse")
plt.xlabel("Temps (s)")
plt.ylim([0.0, 3500])
plt.ylabel("Vitesse (rpm)")
plt.title("100 mesures d'ADC en série")
plt.legend()

In [ ]:
plt.step(time_table,adc_table, label = "Mesure ADC")#, marker = ".")
plt.xlabel("Temps (s)")
plt.ylim([0.0, 3.3])
plt.ylabel("Amplitude (V)")
plt.title("100 mesures d'ADC en série")
plt.legend()